# Hotspot + Position-bin 피처 실험

## 실험 목적
기존 피처 구성 E(아미노산 치환 쌍 + 유전자별 끊김형 피처)를 기준으로
Hotspot 및 Position-bin 피처를 추가하여 성능 변화를 비교한다.

## Baseline
- 기준 실험: 피처 구성 E (아미노산 치환 쌍 + 유전자별 끊김형 피처)
- 모델: LogisticRegression(C=0.1, class_weight="balanced")
- CV Macro F1: 0.4919

## 추가 실험
1. 피처 구성 E + Hotspot
2. 피처 구성 E + Position-bin
3. 피처 구성 E + Hotspot + Position-bin

## 출처
- 전체 근거: Vogelstein et al., Science 2013 / Chang et al., Nat Biotechnol 2016 (cancerhotspots.org)
- IDH1·IDH2: Yan et al., NEJM 2009 / NPM1: Falini et al., NEJM 2005 / EGFR A289·G598: Lee et al., PLoS Med 2006

# 1. 기준 실험 준비하기

In [4]:
import sys
from pathlib import Path

# 프로젝트 루트 경로
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Python 모듈 검색 경로에 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("프로젝트 루트:", PROJECT_ROOT)
print("src 폴더 존재:", (PROJECT_ROOT / "src").is_dir())

프로젝트 루트: /Users/admin/Documents/ah07-cancer-subtype
src 폴더 존재: True


In [5]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags,
)
from src.validation import get_folds, run_cv

print("Imports completed")

Imports completed


In [6]:
# 데이터 로딩 및 팀 공통 Fold 생성
DATA_DIR = PROJECT_ROOT / "data"

train, test = load_data(DATA_DIR)

X, y_raw = split_xy(train)
y, le = encode_target(y_raw)

folds = get_folds(y)

print("train:", train.shape)
print("test:", test.shape)
print("classes:", len(le.classes_))
print("folds:", len(folds))

train: (6201, 4386)
test: (2546, 4385)
classes: 26
folds: 5


In [7]:
# 기준 모델 실행

# 피처 구성 E — 아미노산 치환 쌍 + 유전자별 끊김형 피처

def make_features_E(X):
    X_binary = binarize_mutations(X)

    X_count = np.log1p(mutation_count(X))
    X_types = np.log1p(mutation_type_counts(X))
    X_pairs = np.log1p(amino_acid_pair_counts(X))
    X_trunc = gene_truncation_flags(X)

    return pd.concat(
        [X_binary, X_count, X_types, X_pairs, X_trunc],
        axis=1
    )


def make_features_E_sparse(X):
    return csr_matrix(
        make_features_E(X).to_numpy(dtype=np.float32)
    )


pipeline_E_LR = Pipeline([
    ("features", FunctionTransformer(make_features_E_sparse)),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_E_LR = run_cv(
    pipeline_E_LR,
    X,
    y,
    folds=folds
)

fold 1: Macro F1 = 0.4934
fold 2: Macro F1 = 0.4797
fold 3: Macro F1 = 0.4758
fold 4: Macro F1 = 0.5161
fold 5: Macro F1 = 0.4947
CV Macro F1 = 0.4919 (std 0.0141)


- 기존 팀 기준 점수와 정확히 일치함. 추가 피처의 성능을 동일한 조건에서 비교할 수 있음.

In [32]:
# 유전자별 Hotspot 아미노산 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}

# 일반 Hotspot 35개 + NPM1 frameshift 1개
assert sum(len(v) for v in HOTSPOTS.values()) == 35

print("HOTSPOTS 정의 완료")


n_hotspots = sum(len(v) for v in HOTSPOTS.values()) + 1

print("일반 Hotspot:", sum(len(v) for v in HOTSPOTS.values()))
print("NPM1 frameshift:", 1)
print("전체 Hotspot 피처 수:", n_hotspots)

assert n_hotspots == 36

HOTSPOTS 정의 완료
일반 Hotspot: 35
NPM1 frameshift: 1
전체 Hotspot 피처 수: 36


In [10]:
genes = ["IDH1", "BRAF", "TP53", "NPM1"]

for gene in genes:
    print(f"\n[{gene}]")
    if gene in X.columns:
        print(X[gene].dropna().value_counts().head(10))
    else:
        print("해당 유전자 컬럼 없음")


[IDH1]
IDH1
WT       5760
R132H     345
R132C      46
R132G      10
R132S       8
P33S        2
Q185P       2
R132L       2
I251I       1
Q228*       1
Name: count, dtype: int64

[BRAF]
BRAF
WT             5763
V600E           305
V600K            15
V600E V600E      11
K601E             6
G469R             3
P403fs            3
D594G             3
V600R             3
G466V             2
Name: count, dtype: int64

[TP53]
TP53
WT       4431
R273C      75
R175H      60
R273H      53
R248Q      46
R248W      43
R213*      38
Y220C      32
R282W      27
R196*      26
Name: count, dtype: int64

[NPM1]
NPM1
WT         6145
WQ288fs      21
-287fs       17
V99A          1
K257E         1
W288fs        1
Q114Q         1
P216S         1
L136F         1
E185K         1
Name: count, dtype: int64


In [11]:
def make_hotspot_features(X):
    hotspot_data = {}

    # 일반 Hotspot
    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")

        tokens = X[gene].fillna("WT").astype(str).str.split()

        for position in positions:
            col_name = f"hotspot_{gene}_{position}"

            hotspot_data[col_name] = tokens.map(
                lambda variants: int(
                    any(v.startswith(position) for v in variants)
                )
            ).astype(np.int8)

    # NPM1 frameshift
    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")

    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"]
        .fillna("WT")
        .astype(str)
        .str.contains("fs", case=False, regex=False)
        .astype(np.int8)
    )

    return pd.DataFrame(hotspot_data, index=X.index)


# 생성 결과 확인
X_hotspot = make_hotspot_features(X)

print("Hotspot 피처 shape:", X_hotspot.shape)
print("Hotspot 피처 수:", X_hotspot.shape[1])
print("Hotspot 보유 환자 수:", X_hotspot.any(axis=1).sum())
print("Hotspot별 양성 환자 수 상위 10개:")
print(X_hotspot.sum().sort_values(ascending=False).head(10))

assert X_hotspot.shape == (len(X), 36)
assert X_hotspot.isna().sum().sum() == 0

Hotspot 피처 shape: (6201, 36)
Hotspot 피처 수: 36
Hotspot 보유 환자 수: 1740
Hotspot별 양성 환자 수 상위 10개:
hotspot_IDH1_R132       411
hotspot_BRAF_V600       339
hotspot_PIK3CA_H1047    172
hotspot_TP53_R273       168
hotspot_PIK3CA_E545     163
hotspot_TP53_R248       110
hotspot_PIK3CA_E542      83
hotspot_TP53_R175        77
hotspot_PTEN_R130        63
hotspot_NPM1_fs          40
dtype: int64


# 2. 추가 실험

## 2-1. 피처 구성 E + Hotspot (×1)

In [12]:
# 피처 구성 E + Hotspot
def make_features_E_hotspot(X):
    X_base = make_features_E(X)
    X_hotspot = make_hotspot_features(X)

    return pd.concat(
        [X_base, X_hotspot],
        axis=1
    )


def make_features_E_hotspot_sparse(X):
    return csr_matrix(
        make_features_E_hotspot(X).to_numpy(dtype=np.float32)
    )


pipeline_E_hotspot_LR = Pipeline([
    ("features", FunctionTransformer(make_features_E_hotspot_sparse)),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_E_hotspot_LR = run_cv(
    pipeline_E_hotspot_LR,
    X,
    y,
    folds=folds
)

print(
    "기준 CV:", round(cv_E_LR["mean"], 4),
    "| Hotspot CV:", round(cv_E_hotspot_LR["mean"], 4),
    "| 차이:", round(cv_E_hotspot_LR["mean"] - cv_E_LR["mean"], 4)
)

fold 1: Macro F1 = 0.5020
fold 2: Macro F1 = 0.4828
fold 3: Macro F1 = 0.4764
fold 4: Macro F1 = 0.5121
fold 5: Macro F1 = 0.4981
CV Macro F1 = 0.4943 (std 0.0130)
기준 CV: 0.4919 | Hotspot CV: 0.4943 | 차이: 0.0023


## 2-2. 피처 구성 E + Position-bin

In [ ]:
#  변이 위치 추출 방식 확인

import re

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None

examples = [
    "V600E",
    "R132H",
    "WQ288fs",
    "-287fs",
    "R213*",
    "WT",
    "V600E V600E",
]

for variant in examples:
    tokens = variant.split()
    positions = [extract_position(token) for token in tokens]
    print(f"{variant:15} → {positions}")

V600E           → [600]
R132H           → [132]
WQ288fs         → [288]
-287fs          → [287]
R213*           → [213]
WT              → [None]
V600E V600E     → [600, 600]


In [ ]:
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


rows, cols, gene_bins = extract_position_bins(X)

print("Position-bin 등장 횟수:", len(rows))
print("Position-bin 후보 수:", len(gene_bins))
print("예시:", sorted(gene_bins)[:10])

Position-bin 등장 횟수: 245623
Position-bin 후보 수: 49116
예시: ['A2M_bin_0', 'A2M_bin_1', 'A2M_bin_10', 'A2M_bin_11', 'A2M_bin_12', 'A2M_bin_13', 'A2M_bin_14', 'A2M_bin_15', 'A2M_bin_16', 'A2M_bin_17']


In [15]:
# Fold별 Position-bin 개수 확인

from collections import Counter

# 첫 번째 Fold의 학습 데이터만 사용
tr_idx, va_idx = folds[0]
X_train_fold = X.iloc[tr_idx]

# 학습 환자별 Position-bin 추출
rows_tr, cols_tr, _ = extract_position_bins(X_train_fold)

# 환자별 중복은 이미 제거되어 있으므로 등장 횟수 = 환자 수
bin_counts = Counter(cols_tr)

# 학습 환자 5명 이상 등장한 구간만 선택
selected_bins = {
    feature for feature, count in bin_counts.items()
    if count >= 5
}

print("Fold 1 학습 환자 수:", len(X_train_fold))
print("전체 Position-bin 후보:", len(bin_counts))
print("5명 이상 등장한 Position-bin:", len(selected_bins))
print("선택된 피처 예시:", sorted(selected_bins)[:10])

Fold 1 학습 환자 수: 4961
전체 Position-bin 후보: 47569
5명 이상 등장한 Position-bin: 15123
선택된 피처 예시: ['A2M_bin_1', 'A2M_bin_10', 'A2M_bin_11', 'A2M_bin_14', 'A2M_bin_15', 'A2M_bin_17', 'A2M_bin_19', 'A2M_bin_2', 'A2M_bin_20', 'A2M_bin_21']


In [17]:
# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )

In [ ]:
# 첫 번째 Fold에서 검증

tr_idx, va_idx = folds[0]

position_transformer = PositionBinTransformer(
    bin_size=50,
    min_patients=5
)

X_pos_train = position_transformer.fit_transform(X.iloc[tr_idx])
X_pos_valid = position_transformer.transform(X.iloc[va_idx])

print("학습 Position-bin:", X_pos_train.shape)
print("검증 Position-bin:", X_pos_valid.shape)
print("선택된 피처 수:", len(position_transformer.selected_bins_))

assert X_pos_train.shape[1] == X_pos_valid.shape[1]
assert X_pos_train.shape[1] == 15123
assert X_pos_train.nnz > 0
assert X_pos_valid.nnz > 0

학습 Position-bin: (4961, 15123)
검증 Position-bin: (1240, 15123)
선택된 피처 수: 15123


In [19]:
# 피처 구성 E + Position-bin CV

from scipy.sparse import hstack

class FeaturesEWithPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Fold 학습 데이터에서 선택한 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_position_LR = Pipeline([
    ("features", FeaturesEWithPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_E_position_LR = run_cv(
    pipeline_E_position_LR,
    X,
    y,
    folds=folds
)

print("\n--- CV 비교 ---")
print(f"피처 구성 E 기준: {cv_E_LR['mean']:.4f}")
print(f"피처 구성 E + Position-bin: {cv_E_position_LR['mean']:.4f}")
print(f"차이: {cv_E_position_LR['mean'] - cv_E_LR['mean']:+.4f}")

fold 1: Macro F1 = 0.5070
fold 2: Macro F1 = 0.5046
fold 3: Macro F1 = 0.4847
fold 4: Macro F1 = 0.5118
fold 5: Macro F1 = 0.5020
CV Macro F1 = 0.5020 (std 0.0092)

--- CV 비교 ---
피처 구성 E 기준: 0.4919
피처 구성 E + Position-bin: 0.5020
차이: +0.0101


## 2-3. 피처 구성 E + Hotspot(×1) + Position-bin

In [20]:
# 피처 구성 E + Hotspot + Position-bin 결합 실험

class FeaturesEWithHotspotAndPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        # 피처 구성 E — 아미노산 치환 쌍 + 유전자별 끊김형
        X_base = make_features_E_sparse(X)

        # Hotspot 36개 (일반 35개 + NPM1 frameshift)
        X_hotspot = csr_matrix(
            make_hotspot_features(X).to_numpy(dtype=np.float32)
        )

        # Fold 학습 데이터에서 선택된 Position-bin
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_hotspot, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_hotspot_position_LR = Pipeline([
    ("features", FeaturesEWithHotspotAndPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_E_hotspot_position_LR = run_cv(
    pipeline_E_hotspot_position_LR,
    X,
    y,
    folds=folds
)

print("\n--- 전체 CV 비교 ---")
print(f"피처 구성 E 기준: {cv_E_LR['mean']:.4f}")
print(f"피처 구성 E + Hotspot: {cv_E_hotspot_LR['mean']:.4f}")
print(f"피처 구성 E + Position-bin: {cv_E_position_LR['mean']:.4f}")
print(f"피처 구성 E + Hotspot + Position-bin: {cv_E_hotspot_position_LR['mean']:.4f}")

fold 1: Macro F1 = 0.5112
fold 2: Macro F1 = 0.5072
fold 3: Macro F1 = 0.4905
fold 4: Macro F1 = 0.5131
fold 5: Macro F1 = 0.5051
CV Macro F1 = 0.5054 (std 0.0080)

--- 전체 CV 비교 ---
피처 구성 E 기준: 0.4919
피처 구성 E + Hotspot: 0.4943
피처 구성 E + Position-bin: 0.5020
피처 구성 E + Hotspot + Position-bin: 0.5054


## 2-4. 피처 구성 E + Hotspot (×3)

In [22]:
def make_features_E_hotspot3_sparse(X):
    X_base = make_features_E_sparse(X)

    X_hotspot = csr_matrix(
        make_hotspot_features(X).to_numpy(dtype=np.float32) * 3
    )

    return hstack(
        [X_base, X_hotspot],
        format="csr",
        dtype=np.float32
    )


pipeline_E_hotspot3_LR = Pipeline([
    ("features", FunctionTransformer(make_features_E_hotspot3_sparse)),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_E_hotspot3_LR = run_cv(
    pipeline_E_hotspot3_LR,
    X,
    y,
    folds=folds
)

print("\n--- 피처 구성 E + Hotspot 비교 ---")
print(f"Hotspot ×1: {cv_E_hotspot_LR['mean']:.4f}")
print(f"Hotspot ×3: {cv_E_hotspot3_LR['mean']:.4f}")
print(f"차이: {cv_E_hotspot3_LR['mean'] - cv_E_hotspot_LR['mean']:+.4f}")

fold 1: Macro F1 = 0.5000
fold 2: Macro F1 = 0.4928
fold 3: Macro F1 = 0.4861
fold 4: Macro F1 = 0.5216
fold 5: Macro F1 = 0.5000
CV Macro F1 = 0.5001 (std 0.0119)

--- 피처 구성 E + Hotspot 비교 ---
Hotspot ×1: 0.4943
Hotspot ×3: 0.5001
차이: +0.0058


## 2-5. 피처 구성 E + Hotspot(×3) + Position-bin

In [23]:
class FeaturesEWithHotspot3AndPositionBins(
    FeaturesEWithHotspotAndPositionBins
):
    def transform(self, X):
        # 기존 피처 구성 E
        X_base = make_features_E_sparse(X)

        # Hotspot 36개만 ×3
        X_hotspot = csr_matrix(
            make_hotspot_features(X).to_numpy(dtype=np.float32) * 3
        )

        # Position-bin은 기존 0/1 유지
        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_hotspot, X_position],
            format="csr",
            dtype=np.float32
        )


pipeline_E_hotspot3_position_LR = Pipeline([
    ("features", FeaturesEWithHotspot3AndPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_E_hotspot3_position_LR = run_cv(
    pipeline_E_hotspot3_position_LR,
    X,
    y,
    folds=folds
)

print("\n========== 피처 구성 E + Hotspot + Position-bin 비교 ==========")

print(f"기준 E                            : {cv_E_LR['mean']:.4f}")
print(f"E + Hotspot(×1) + Position-bin     : {cv_E_hotspot_position_LR['mean']:.4f}")
print(f"E + Hotspot(×3) + Position-bin     : {cv_E_hotspot3_position_LR['mean']:.4f}")

print("\n--- 성능 차이 ---")
print(
    f"기준 E 대비 (Hotspot ×3) : "
    f"{cv_E_hotspot3_position_LR['mean'] - cv_E_LR['mean']:+.4f}"
)
print(
    f"Hotspot ×1 대비 ×3      : "
    f"{cv_E_hotspot3_position_LR['mean'] - cv_E_hotspot_position_LR['mean']:+.4f}"
)

fold 1: Macro F1 = 0.5142
fold 2: Macro F1 = 0.5117
fold 3: Macro F1 = 0.4952
fold 4: Macro F1 = 0.5210
fold 5: Macro F1 = 0.5003
CV Macro F1 = 0.5085 (std 0.0094)

========== 피처 구성 E + Hotspot + Position-bin 비교 ==========
기준 E                            : 0.4919
E + Hotspot(×1) + Position-bin     : 0.5054
E + Hotspot(×3) + Position-bin     : 0.5085

--- 성능 차이 ---
기준 E 대비 (Hotspot ×3) : +0.0165
Hotspot ×1 대비 ×3      : +0.0030


# 3. 실험 결과 CSV 저장

In [24]:
# 실험 결과 정리
experiment_results = pd.DataFrame([
    {
        "experiment": "E_baseline",
        "hotspot_scale": 0,
        "position_bin": False,
        "cv_macro_f1": cv_E_LR["mean"],
        "cv_std": cv_E_LR["std"],
    },
    {
        "experiment": "E_hotspot_x1",
        "hotspot_scale": 1,
        "position_bin": False,
        "cv_macro_f1": cv_E_hotspot_LR["mean"],
        "cv_std": cv_E_hotspot_LR["std"],
    },
    {
        "experiment": "E_hotspot_x3",
        "hotspot_scale": 3,
        "position_bin": False,
        "cv_macro_f1": cv_E_hotspot3_LR["mean"],
        "cv_std": cv_E_hotspot3_LR["std"],
    },
    {
        "experiment": "E_position_bin",
        "hotspot_scale": 0,
        "position_bin": True,
        "cv_macro_f1": cv_E_position_LR["mean"],
        "cv_std": cv_E_position_LR["std"],
    },
    {
        "experiment": "E_hotspot_x1_position_bin",
        "hotspot_scale": 1,
        "position_bin": True,
        "cv_macro_f1": cv_E_hotspot_position_LR["mean"],
        "cv_std": cv_E_hotspot_position_LR["std"],
    },
    {
        "experiment": "E_hotspot_x3_position_bin",
        "hotspot_scale": 3,
        "position_bin": True,
        "cv_macro_f1": cv_E_hotspot3_position_LR["mean"],
        "cv_std": cv_E_hotspot3_position_LR["std"],
    },
])

# 실험 설정 기록
experiment_results["model"] = "LogisticRegression"
experiment_results["C"] = 0.1
experiment_results["class_weight"] = "balanced"
experiment_results["n_folds"] = 5
experiment_results["bin_size"] = 50
experiment_results["min_patients"] = 5

# 성능순 정렬
experiment_results = experiment_results.sort_values(
    "cv_macro_f1", ascending=False
)

# CSV 저장
save_path = PROJECT_ROOT / "experiments" / "H_hotspot_position_results.csv"
save_path.parent.mkdir(parents=True, exist_ok=True)

experiment_results.to_csv(save_path, index=False)

print(f"저장 완료: {save_path}")
display(experiment_results)

저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/H_hotspot_position_results.csv


,experiment,hotspot_scale,position_bin,cv_macro_f1,cv_std,model,C,class_weight,n_folds,bin_size,min_patients
5,E_hotspot_x3_position_bin,3,True,0.508463,0.009404,LogisticRegression,0.1,balanced,5,50,5
4,E_hotspot_x1_position_bin,1,True,0.505419,0.007977,LogisticRegression,0.1,balanced,5,50,5
3,E_position_bin,0,True,0.501997,0.009238,LogisticRegression,0.1,balanced,5,50,5
2,E_hotspot_x3,3,False,0.500108,0.011922,LogisticRegression,0.1,balanced,5,50,5
1,E_hotspot_x1,1,False,0.494282,0.012995,LogisticRegression,0.1,balanced,5,50,5
0,E_baseline,0,False,0.491935,0.014135,LogisticRegression,0.1,balanced,5,50,5


# 4. OOF 예측 확률 저장

In [25]:
import numpy as np

# OOF 예측 확률 저장 폴더
pred_dir = PROJECT_ROOT / "experiments" / "preds"
pred_dir.mkdir(parents=True, exist_ok=True)

# 실험별 CV 결과
oof_results = {
    "E_baseline": cv_E_LR,
    "E_hotspot_x1": cv_E_hotspot_LR,
    "E_hotspot_x3": cv_E_hotspot3_LR,
    "E_position_bin": cv_E_position_LR,
    "E_hotspot_x1_position_bin": cv_E_hotspot_position_LR,
    "E_hotspot_x3_position_bin": cv_E_hotspot3_position_LR,
}

# OOF 저장
for experiment_name, result in oof_results.items():

    oof = result["oof_proba"]

    # 데이터 크기 및 클래스 수 검증
    assert oof.shape == (len(X), len(le.classes_))
    assert np.isfinite(oof).all()

    save_path = pred_dir / f"H_data__{experiment_name}__oof.npy"

    np.save(save_path, oof)

    print(f"{experiment_name}: {oof.shape} 저장 완료")

E_baseline: (6201, 26) 저장 완료
E_hotspot_x1: (6201, 26) 저장 완료
E_hotspot_x3: (6201, 26) 저장 완료
E_position_bin: (6201, 26) 저장 완료
E_hotspot_x1_position_bin: (6201, 26) 저장 완료
E_hotspot_x3_position_bin: (6201, 26) 저장 완료


In [26]:
import inspect
import src.prediction as prediction

print("=== src/prediction.py에 정의된 함수 ===")

for name, obj in inspect.getmembers(prediction, inspect.isfunction):
    if obj.__module__ == prediction.__name__:
        print(f"\n함수명: {name}")
        print(f"매개변수: {inspect.signature(obj)}")

=== src/prediction.py에 정의된 함수 ===

함수명: make_submission
매개변수: (test: pandas.DataFrame, pred_labels, file_name: str, data_dir=PosixPath('/Users/admin/Documents/ah07-cancer-subtype/data')) -> pandas.DataFrame


In [27]:
from sklearn.base import clone

def predict_test_5fold(pipeline, X, y, X_test, folds):
    test_probas = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
        model = clone(pipeline)

        # 해당 Fold의 학습 데이터로만 학습
        model.fit(X.iloc[tr_idx], y[tr_idx])

        # test 데이터의 클래스별 예측 확률
        proba = model.predict_proba(X_test)

        test_probas.append(proba)

        print(f"Fold {fold_num}: test 예측 완료 {proba.shape}")

    # 5개 Fold 예측 확률 평균
    return np.mean(test_probas, axis=0)


# test에서 ID와 타깃 등 비피처 열 제외
X_test = test.drop(columns=["ID", "SUBCLASS"], errors="ignore")

test_proba_best = predict_test_5fold(
    pipeline_E_hotspot3_position_LR,
    X,
    y,
    X_test,
    folds
)

print("\n최종 test 예측 확률:", test_proba_best.shape)

assert test_proba_best.shape == (len(test), len(le.classes_))
assert np.isfinite(test_proba_best).all()
assert np.allclose(test_proba_best.sum(axis=1), 1.0)

print("검증 통과")

Fold 1: test 예측 완료 (2546, 26)
Fold 2: test 예측 완료 (2546, 26)
Fold 3: test 예측 완료 (2546, 26)
Fold 4: test 예측 완료 (2546, 26)
Fold 5: test 예측 완료 (2546, 26)

최종 test 예측 확률: (2546, 26)
검증 통과


In [28]:
# 최고 성능 모델의 test 예측 확률 저장

test_save_path = (
    pred_dir / "H_data__E_hotspot_x3_position_bin__test.npy"
)

np.save(test_save_path, test_proba_best)

# 저장된 파일을 다시 읽어 검증
saved_test = np.load(test_save_path)

assert saved_test.shape == (2546, 26)
assert np.allclose(saved_test, test_proba_best)

print(f"저장 완료: {test_save_path}")
print(f"저장된 배열 크기: {saved_test.shape}")

저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/preds/H_data__E_hotspot_x3_position_bin__test.npy
저장된 배열 크기: (2546, 26)


In [30]:
# # 5개 실험의 test 예측 확률 저장

# remaining_models = {
#     "E_baseline": pipeline_E_LR,
#     "E_hotspot_x1": pipeline_E_hotspot_LR,
#     "E_hotspot_x3": pipeline_E_hotspot3_LR,
#     "E_position_bin": pipeline_E_position_LR,
#     "E_hotspot_x1_position_bin": pipeline_E_hotspot_position_LR,
# }

# # 학습 데이터와 test 피처 순서 일치
# X_test = test[X.columns].copy()

# for experiment_name, pipeline in remaining_models.items():
#     print(f"\n===== {experiment_name} =====")

#     test_proba = predict_test_5fold(
#         pipeline,
#         X,
#         y,
#         X_test,
#         folds
#     )

#     # 예측 확률 검증
#     assert test_proba.shape == (len(test), len(le.classes_))
#     assert np.isfinite(test_proba).all()
#     assert np.allclose(test_proba.sum(axis=1), 1.0)

#     # 파일 저장
#     save_path = pred_dir / f"H_data__{experiment_name}__test.npy"
#     np.save(save_path, test_proba)

#     print(f"저장 완료: {save_path.name}")

# print("\n나머지 5개 실험의 test 예측 확률 저장 완료!")

In [31]:
print("HOTSPOTS 존재 여부:", "HOTSPOTS" in globals())

if "HOTSPOTS" in globals():
    print("HOTSPOTS =", repr(HOTSPOTS))

HOTSPOTS 존재 여부: True
HOTSPOTS = {'IDH1': ['R132'], 'IDH2': ['R140', 'R172'], 'BRAF': ['V600'], 'HRAS': ['Q61', 'G13', 'G12'], 'KIT': ['D816', 'N822', 'K642'], 'CTNNB1': ['S33', 'S37', 'S45', 'T41', 'G34', 'D32'], 'U2AF1': ['S34'], 'PIK3CA': ['E545', 'H1047', 'E542'], 'EGFR': ['L858', 'G598', 'A289'], 'RET': ['M918'], 'RAC1': ['P29'], 'SPOP': ['F133', 'W131', 'F102', 'Y87'], 'EZH2': ['Y641', 'Y646'], 'TP53': ['R175', 'R248', 'R273'], 'PTEN': ['R130']}
